# E22 — o que a soma guarda

Este caderno mede a conta que todo mundo faz primeiro, e que erra: somar as variações diárias
de um índice para saber o que o índice andou.

**A tentativa.** Somar as porcentagens de cada dia e comparar com a variação de verdade, a que
sai do preço do fim dividido pelo do começo. Num mês as duas quase coincidem; em vinte e seis
anos elas se separam de muito, e nada na conta avisa.

**O que se mede.**

1. a soma das porcentagens, a variação real e a soma dos logaritmos, em três horizontes;
2. o **desvio do nível** depois de h dias, medido em milhares de mundos sorteados, contra a
   previsão da raiz (o desvio de um dia vezes a raiz de h);
3. a mesma dispersão medida **na série real**, que é onde a previsão se comporta de outro jeito.

In [1]:
# <- brinque com: SERIE, HORIZONTES, MUNDOS, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, nivel, proporcao, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"      # a série do arquivo do projeto anterior (.old/dados/)
CURTA = 21               # um mês de pregões
LONGA = 252              # um ano de pregões
HORIZONTES = (21, 63, 252, 1260)   # um mês, um trimestre, um ano, cinco anos
MUNDOS = 4000            # quantos mundos sorteados por horizonte
SEMENTE = 22

precos = dados.carregar_serie(SERIE)
variacoes = proporcao.variacao(precos)
retornos = volatilidade.retornos_log(precos)
desvio_diario = float(retornos.std(ddof=1))
print("frevolab %s | %s: %d pregões | desvio de um dia: %.5f" % (
    frevolab.VERSAO, SERIE, len(precos), desvio_diario))

frevolab 0.1.0 | sp500.csv: 6719 pregões | desvio de um dia: 0.01213


## A soma das porcentagens contra a variação real

As três contas em cima do mesmo pedaço de série: a soma crua das porcentagens, a variação que
de fato aconteceu e a soma dos logaritmos.

In [2]:
# As três contas, em três horizontes --- a conta mora na biblioteca, e não aqui.
linhas = [dict(horizonte="a série inteira", **nivel.contas_do_pedaco(precos)),
          dict(horizonte="um ano", **nivel.contas_do_pedaco(precos, LONGA)),
          dict(horizonte="um mês", **nivel.contas_do_pedaco(precos, CURTA))]
tabela = pd.DataFrame(linhas).set_index("horizonte")
print(tabela.round(3).to_string())
print()
inteira = nivel.contas_do_pedaco(precos)
print("em %d dias a soma das porcentagens entrega %.1f%% e o índice andou %.1f%%: %.2f vezes" % (
    inteira["dias"], inteira["soma das porcentagens (%)"], inteira["variação real (%)"],
    inteira["variação real (%)"] / inteira["soma das porcentagens (%)"]))

                 dias  soma das porcentagens (%)  variação real (%)  soma dos logaritmos (%)  log do quociente (%)
horizonte                                                                                                         
a série inteira  6718                    216.665            432.790                  167.296               167.296
um ano            252                     16.462             16.908                   15.621                15.621
um mês             21                      1.499              1.467                    1.457                 1.457

em 6718 dias a soma das porcentagens entrega 216.7% e o índice andou 432.8%: 2.00 vezes


## O passeio: o desvio do nível contra a raiz do horizonte

A segunda leitura é sobre o tamanho, e não sobre a soma: depois de h dias, de quanto o nível se
afastou? A previsão é o desvio de um dia vezes a raiz de h, e ela se confere de dois jeitos ---
em mundos sorteados e na própria série.

In [3]:
# A previsão contra os mundos sorteados, e contra a série real.
sorteio = np.random.default_rng(SEMENTE)
previsto, medido, real = {}, {}, {}
for h in HORIZONTES:
    previsto[h] = nivel.desvio_do_nivel(desvio_diario, h)
    medido[h] = float(nivel.mundos_do_passeio(desvio_diario, h, MUNDOS, sorteio).std(ddof=1))
    real[h] = float(nivel.mudancas_de_nivel(precos, h).std(ddof=1))

print("%8s %12s %12s %10s %12s %10s" % ("dias", "previsto(%)", "mundos(%)", "razão",
                                        "série(%)", "razão"))
for h in HORIZONTES:
    print("%8d %12.2f %12.2f %10.3f %12.2f %10.3f" % (
        h, 100 * previsto[h], 100 * medido[h], medido[h] / previsto[h],
        100 * real[h], real[h] / previsto[h]))
print()
print("nos mundos a previsão acerta; na série real ela exagera, e exagera mais quanto mais")
print("longe o horizonte --- porque o desvio de um dia é carregado por poucos dias.")

    dias  previsto(%)    mundos(%)      razão     série(%)      razão
      21         5.56         5.55      0.998         4.81      0.866
      63         9.63         9.57      0.994         7.85      0.815
     252        19.25        19.62      1.019        16.98      0.882
    1260        43.05        43.19      1.003        30.58      0.710

nos mundos a previsão acerta; na série real ela exagera, e exagera mais quanto mais
longe o horizonte --- porque o desvio de um dia é carregado por poucos dias.


In [4]:
# A causa do exagero, isolada: a ORDEM dos dias. Baralhando os retornos, os dias enormes
# continuam todos la e o efeito desaparece --- o que a cauda pesada nao explica e a reversao
# de um dia, e e ela que a razao mede.
rho_um = float(pd.Series(retornos).autocorr(1))
gerador = np.random.default_rng(SEMENTE + 1)
baralhado = {}
for h in HORIZONTES:
    razoes = []
    for _ in range(200):
        r = np.array(retornos)
        gerador.shuffle(r)
        precos_b = pd.Series(float(precos.iloc[0]) * np.exp(np.cumsum(r)), index=precos.index[1:])
        razoes.append(float(nivel.mudancas_de_nivel(precos_b, h).std(ddof=1)) / previsto[h])
    baralhado[h] = (float(np.mean(razoes)), float(np.std(razoes, ddof=1)), len(razoes))
print("autocorrelacao de um dia: %+.4f" % rho_um)
print("%8s %16s %14s" % ("dias", "razao baralhada", "dispersao"))
for h in HORIZONTES:
    print("%8d %16.3f %14.4f" % (h, baralhado[h][0], baralhado[h][1]))


autocorrelacao de um dia: -0.0990
    dias  razao baralhada      dispersao
      21            0.997         0.0302
      63            0.993         0.0552
     252            0.970         0.1047
    1260            0.841         0.2475


## As figuras

In [5]:
# Figura 1: as duas somas, dia a dia, ao longo de vinte e seis anos.
acumulada_pct = 100 * variacoes.cumsum()
real_pct = 100 * (precos / float(precos.iloc[0]) - 1.0)
real_pct = real_pct.reindex(acumulada_pct.index)

fig, eixo = plt.subplots(figsize=(9.2, 4.2))
eixo.plot(acumulada_pct.index, acumulada_pct.to_numpy(), lw=1.4, color="#1f4e79",
          label="a soma das porcentagens: %.1f%%" % float(acumulada_pct.iloc[-1]))
eixo.plot(real_pct.index, real_pct.to_numpy(), lw=1.4, color="#b03a2e",
          label="o que o índice andou: %.1f%%" % float(real_pct.iloc[-1]))
eixo.set_xlabel("ano")
eixo.set_ylabel("variação acumulada (%)")
eixo.set_title("A conta que soma porcentagens, contra o índice")
eixo.legend()
graficos.salvar(fig, "E22_soma", 1)
plt.close(fig)

In [6]:
# Figura 2: o desvio do nível contra o horizonte, nas três leituras.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
eixo.loglog(HORIZONTES, [100 * previsto[h] for h in HORIZONTES], "o-", color="#1f4e79",
            label="a previsão: desvio de um dia vezes raiz de h")
eixo.loglog(HORIZONTES, [100 * medido[h] for h in HORIZONTES], "s--", color="#2e7d32",
            label="medido em %d mundos sorteados" % MUNDOS)
eixo.loglog(HORIZONTES, [100 * real[h] for h in HORIZONTES], "^:", color="#b03a2e",
            label="medido na série real")
eixo.set_xlabel("dias no horizonte")
eixo.set_ylabel("desvio do nível (pontos percentuais)")
eixo.set_title("O desvio do nível, previsto e medido")
eixo.grid(True, which="both", ls=":", lw=0.6, alpha=0.6)
eixo.legend()
graficos.salvar(fig, "E22_soma", 2)
plt.close(fig)

## Leitura visual das figuras

A conferir nesta sessão: as figuras foram abertas em .png pela ponte de visão, e o que segue é
observação --- observação não vira número.

**Figura 1.**

**Figura 2.**

In [7]:
# O resultado: um objeto por grandeza, em português, para o livro citar por comando.
nome = {21: "vinte_um", 63: "sessenta_e_tres", 252: "duzentos_e_cinquenta_e_dois",
        1260: "mil_duzentos_e_sessenta"}
curta = nivel.contas_do_pedaco(precos, CURTA)
longa = nivel.contas_do_pedaco(precos, LONGA)
resultado = {
    "soma_serie_dias": int(len(precos)),
    "soma_variacoes_pct": inteira["soma das porcentagens (%)"],
    "soma_variacao_real_pct": inteira["variação real (%)"],
    "soma_logs_pct": inteira["soma dos logaritmos (%)"],
    "soma_curta_dias": CURTA,
    "soma_curta_variacoes_pct": curta["soma das porcentagens (%)"],
    "soma_curta_real_pct": curta["variação real (%)"],
    "soma_longa_dias": LONGA,
    "soma_longa_variacoes_pct": longa["soma das porcentagens (%)"],
    "soma_longa_real_pct": longa["variação real (%)"],
    "soma_desvio_diario_pct": 100 * desvio_diario,
    "soma_rho_um": rho_um,
    "soma_baralhada_razao_vinte_um": baralhado[21][0],
    "soma_baralhada_razao_vinte_um_dispersao": baralhado[21][1],
    "soma_baralhada_sorteios": baralhado[21][2],
    "soma_baralhada_razao_mil_duzentos_e_sessenta": baralhado[1260][0],
    "soma_baralhada_razao_mil_duzentos_e_sessenta_dispersao": baralhado[1260][1],
    "soma_mundos": MUNDOS,
}
for h in HORIZONTES:
    resultado["soma_passeio_%s_previsto_pct" % nome[h]] = 100 * previsto[h]
    resultado["soma_passeio_%s_medido_pct" % nome[h]] = 100 * medido[h]
    resultado["soma_passeio_%s_razao" % nome[h]] = medido[h] / previsto[h]
    resultado["soma_real_%s_dispersao_pct" % nome[h]] = 100 * real[h]
    resultado["soma_real_%s_razao" % nome[h]] = real[h] / previsto[h]
    resultado["soma_horizonte_%s_dias" % nome[h]] = h

caminho = Path("lab/resultados/E22_soma.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E22_soma.json gravado | 42 grandezas
